# book

> a book of verses read into chapters, through vishalakshi: every page by a vision model, verses gathered, numbers mended, akṣaras repaired, counts checked

In [ ]:
#| default_exp book

In [ ]:
#| hide
from fastcore.test import *
import tempfile, json

In [ ]:
#| export
from fastcore.all import AttrDict, L, Path
from vishalakshi import read_doc
from lekhaka.verses import assemble, check_counts, fix_aksharas, bad_aksharas
from lekhaka.repairs import apply_fixes

`read_book` reads a PDF or page image of a book of verses. Each page goes to `model` with the `preset` reading prompt (`sanskrit` by default), so it comes back as blocks: verses with their printed numbers, headings, colophons. With a `vault` the pages are kept there as versions, under one document, and `doc_id` says which; without one nothing is stored. The verses are assembled into chapters (`assemble`) and their akṣaras repaired when `fix`. `kinds`, `notes` and `number_from_note` go to `assemble`; `fixes`, a table for `apply_fixes`, is applied first. `counts`, the verses each chapter should hold, adds what differs to `probs`. `chat` stands in for the model, as the tests do.

In [ ]:
#| export
def read_book(path, vault=None, model:str=None, preset:str='sanskrit', title:str='', script:str='Devanagari', hint:str='',
              pages=None, counts=None, fix:bool=True, pair:bool=False, kinds=('verse', None), notes=(), number_from_note:bool=False,
              fixes=None, dpi:int=300, workers:int=4, chat=None) -> AttrDict:
    "A book's chapters `[{heading, colophon, verses}]`, the problems a proofreader should see, and the pages they came from."
    kw = dict(parse='llm', model=model, preset=preset, script=script, hint=hint, pages=pages, dpi=dpi, workers=workers, chat=chat)
    if vault is not None:
        o = vault.read_doc(path, title=title or None, **kw)
        did, ps = o['doc_id'], vault.pages(o['doc_id'])
    else: did, ps = None, read_doc(path, title=title, **kw)
    ps = [dict(page=p['page'], blocks=p.get('blocks') or []) for p in ps if pages is None or p['page'] in set(pages)]
    probs = []
    if fixes: apply_fixes(ps, fixes, probs=probs)
    chs, more = assemble(ps, pair, kinds, notes, number_from_note); probs += more
    for c in chs:
        for v in c['verses']:
            if fix and script == 'Devanagari' and bad_aksharas(v['text']): v['text'] = fix_aksharas(v['text'])
    if counts: probs += check_counts(chs, counts)
    return AttrDict(doc_id=did, chapters=chs, probs=probs, pages=len(ps))

def book_md(chapters, title:str='') -> str:
    "Chapters as markdown: a heading per chapter, each verse with its number, the colophon after."
    out = [f'# {title}'] if title else []
    for i, c in enumerate(chapters, 1):
        out += ['', f"## {c['heading'] or f'Chapter {i}'}", '']
        out += [f"{v['text']}  \n*{v['n']}*\n" for v in c['verses']]
        if c['colophon']: out += [f"*{c['colophon']}*"]
    return '\n'.join(out).strip() + '\n'

In [ ]:
from pdflite import pdf_doc
_src = Path(tempfile.mkdtemp())/'two.pdf'
from pdf_oxide import Pdf
_fx = Path(tempfile.mkdtemp())
for _i, _t in enumerate(['# Attention Is All You Need\n\nThe dominant sequence transduction models are based on recurrent or convolutional networks.',
                         '# Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.']):
    Pdf.from_markdown(_t).save(str(_fx/f'p{_i}.pdf'))
Pdf.merge([str(_fx/'p0.pdf'), str(_fx/'p1.pdf')]).save(str(_fx/'two.pdf'))
_att = _fx/'two.pdf'
_src.write_bytes(pdf_doc(pdf_doc(_att).extract_pages_to_bytes([0, 1])).flatten_to_images(dpi=60))
_reads = iter([
    dict(items=[dict(kind='heading', text='अथ प्रथमोऽध्यायः'), dict(kind='verse', n='१', text='धर्मक्षेत्रे कुरुक्षेत्रे ।\nसमवेता युयुत्सवः ॥१॥'),
                dict(kind='verse', n=None, text='मामकाः पाण्डवाश्चैव ।')]),
    dict(items=[dict(kind='verse', n='॥२॥', text='किमकुर्वत सञ्जय ॥२॥'), dict(kind='colophon', text='इति प्रथमोऽध्यायः')])])
class Book:
    "Reads the two pages in order, whatever order the threads ask in."
    model, lock = 'fake/vision', __import__('threading').Lock()
    def __call__(self, msg):
        with Book.lock: return dict(content=json.dumps(next(_reads), ensure_ascii=False))
_b = read_book(_src, chat=Book(), workers=1)
test_eq(_b.pages, 2); test_eq(_b.doc_id, None); test_eq(_b.probs, [])
_c = _b.chapters[0]
test_eq((_c['heading'], _c['colophon']), ('अथ प्रथमोऽध्यायः', 'इति प्रथमोऽध्यायः'))
test_eq([v['n'] for v in _c['verses']], [1, 2]); assert _c['verses'][1]['text'].startswith('मामकाः')
_md = book_md(_b.chapters, 'Gita')
assert _md.startswith('# Gita') and '## अथ प्रथमोऽध्यायः' in _md and '*2*' in _md
test_eq(check_counts(_b.chapters, [3]), ['chapter 1: 2 verses, the plan has 3'])

In [ ]:
from vishalakshi import Vault
_reads = iter([dict(items=[dict(kind='verse', n=1, text='क ॥१॥')]), dict(items=[dict(kind='verse', n=2, text='ध्ार्म ॥२॥')])])
_v = Vault(Path(tempfile.mkdtemp())/'v.db', offline=True)
_vb = read_book(_src, _v, chat=Book(), workers=1)
assert _vb.doc_id and len(_v.pages(_vb.doc_id)) == 2
test_eq([v['text'] for v in _vb.chapters[0]['verses']], ['क ॥१॥', 'धार्म ॥२॥'])   # repaired

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()